## Task 5: Auto-Tagging Support Tickets Using an LLM

**Objective:** Automatically assign the top 3 most relevant tags to free-text support tickets.

**Approach:**
1. Zero-Shot Prompting — no examples given to the LLM
2. Few-Shot Prompting — give 5 labeled examples inside the prompt
3. Output top 3 tags per ticket

### 1. Importing required libraries


In [1]:
import pandas as pd
import time
from groq import Groq # Note that I'm using Groq instead of HuggingFace (cz of some issues with huggingface)
from sklearn.metrics import classification_report, accuracy_score

In [8]:
# GROQ API key
MODEL_NAME = "openai/gpt-oss-120b"
API_KEY = "My API key"
client = Groq(api_key = API_KEY)

### 2. Working with dataset

In [3]:
df = pd.read_csv("customer_support_tickets.csv")

testDF = df.head(50) # I'm just using the first 50 rows for testing
testDF = testDF[["Ticket Description", "Ticket Type"]]
testDF.head()

,Ticket Description,Ticket Type
0,I'm having an issue with the {product_purchase...,Technical issue
1,I'm having an issue with the {product_purchase...,Technical issue
2,I'm facing a problem with my {product_purchase...,Technical issue
3,I'm having an issue with the {product_purchase...,Billing inquiry
4,I'm having an issue with the {product_purchase...,Billing inquiry


### 3. Classification 

In [4]:
def classify_ticket(prompt, mode="few-shot"):
    categories = testDF['Ticket Type'].unique().tolist()
    categoriesAsString = ', '.join(categories)
    
    # Mode = few shot by default but can be changed to zero-shot by passing "zero-shot" as an argument
    if mode == "zero-shot":
        system_instructions = f"""
        Categorize this ticket into the top 3 tags from: {categories}.
        'Rules:\n'
        '- Only use tags from the list {categoriesAsString}. Do not create new tags.\n'
        '- Return exactly 3 tags, ordered from most to least relevant.\n'
        '- Respond ONLY with tag names separated by commas.\n'
        '- Format: Tag1, Tag2, Tag3\n\n'
        """ # No examples
    else:
        example_rows = testDF.head(5) # Using first 5 rows as examples for few-shot prompting

        examples_text = "" # Examples text initialized
        for _, row in example_rows.iterrows():
            examples_text += f"Ticket: '{row['Ticket Description'][:60]}' -> {row['Ticket Type']}\n" # Using only first 60 characters of the ticket description in examples
        system_instructions = f"""
        'Rules:\n'
        '- Only use tags from the list {categoriesAsString}. Do not create new tags.\n'
        '- Return exactly 3 tags, ordered from most to least relevant.\n'
        '- Respond ONLY with tag names separated by commas.\n'
        '- Format: Tag1, Tag2, Tag3\n\n'
        Categorize the ticket into top 3 tags from: {categories}.
        Here are some examples:
        {examples_text}
        """ # Few examples given

    # Calling the model thorugh API
    response = client.chat.completions.create(
        messages=[
            {"role": "system", "content": system_instructions},
            {"role": "user", "content": prompt}
        ],
        model = MODEL_NAME,
        temperature = 0.6
    )
    
    return response.choices[0].message.content

### 4. Running predicitons

In [6]:
# Lists to store our results for evaluation
y_true = [] # Actual categories from CSV
y_pred = [] # Top-1 prediction from AI

print("Starting Classification...")

for index, row in testDF.iterrows():
    # Get the AI output (Top 3 tags)
    full_output = classify_ticket(row['Ticket Description'])
    
    # Extract the very first tag as the 'Primary' prediction for scikit-learn
    primary_prediction = full_output.split(',')[0].strip()
    
    # Store results
    y_true.append(row['Ticket Type'])
    y_pred.append(primary_prediction)
    
    print(f"Row {index+1}: Done")
    print(f"Predicted top-3: {full_output} \t Actual: {row['Ticket Type']}")
    time.sleep(2) # Pause to avoid Rate Limit errors

Starting Classification...
Row 1: Done
Predicted top-3: Technical issue, Billing inquiry, Product inquiry 	 Actual: Technical issue
Row 2: Done
Predicted top-3: Technical issue, Product inquiry, Billing inquiry 	 Actual: Technical issue
Row 3: Done
Predicted top-3: Technical issue, Product inquiry, Billing inquiry 	 Actual: Technical issue
Row 4: Done
Predicted top-3: Technical issue, Product inquiry, Billing inquiry 	 Actual: Billing inquiry
Row 5: Done
Predicted top-3: Technical issue, Product inquiry, Refund request 	 Actual: Billing inquiry
Row 6: Done
Predicted top-3: Technical issue, Product inquiry, Refund request 	 Actual: Cancellation request
Row 7: Done
Predicted top-3: Technical issue, Product inquiry, Billing inquiry 	 Actual: Product inquiry
Row 8: Done
Predicted top-3: Technical issue, Product inquiry, Billing inquiry 	 Actual: Refund request
Row 9: Done
Predicted top-3: Technical issue, Product inquiry, Billing inquiry 	 Actual: Technical issue
Row 10: Done
Predicted top

### 5. Evaluation using scikit-learn

In [7]:
# Calculate Accuracy
score = accuracy_score(y_true, y_pred)
print(f"Overall Accuracy: {score * 100:.2f}%")

# Generate detailed report (Precision, Recall, F1)
print("\nDetailed Classification Report:")
print(classification_report(y_true, y_pred))

Overall Accuracy: 26.00%

Detailed Classification Report:
                      precision    recall  f1-score   support

     Billing inquiry       0.00      0.00      0.00         8
Cancellation request       0.00      0.00      0.00         8
     Product inquiry       0.00      0.00      0.00        12
      Refund request       0.00      0.00      0.00         9
     Technical issue       0.26      1.00      0.41        13

            accuracy                           0.26        50
           macro avg       0.05      0.20      0.08        50
        weighted avg       0.07      0.26      0.11        50



c:\Users\Muhammad Noman\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Muhammad Noman\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Muhammad Noman\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control thi